# 1. pgvector(PostgreSQL 확장) Setting

In [ ]:
import os
from dotenv import load_dotenv
import psycopg2

load_dotenv()

POSTGRESQL_PORT = os.getenv("POSTGRESQL_PORT")
POSTGRESQL_USER = os.getenv("POSTGRESQL_USER")
POSTGRESQL_PASSWORD = os.getenv("POSTGRESQL_PASSWORD")
POSTGRESQL_DBNAME = os.getenv("POSTGRESQL_DBNAME")


In [ ]:
# postgreSQL 접속 테스트
conn = psycopg2.connect(
    host="localhost",   # 클라우드였다면 VM 공인 IP
    port=POSTGRESQL_PORT,
    user=POSTGRESQL_USER,
    password=POSTGRESQL_PASSWORD,
    dbname=POSTGRESQL_DBNAME
)

## Table 설계
- id : primary key용
- qa_id : 파일명 (qa_data_qa_id 값)

- instruction : 요구사항 (qa_data_instruction)
- question : 고객질문 (qa_data_input_question) 
- answer : 상담사답변 (qa_data_input_answer) 
- follow_up_question : 꼬리질문 (qa_data_input_follow_up_question)
- output : 종합답변 (qa_data_output)

- full_source : “요구사항:~\n고객질문:~\n상담사답변:~\n꼬리질문:~\n종합답변:~”꼴로 이루어진 전처리값

- consulting_category
- consulting_topic
- qa_topic
- consulting_purpose

- embedding_q : question에 대한 임베딩 값
- embedding_i : instruction에 대한 임베딩 값
- embedding_full : full_source에 대한 임베딩 값


In [ ]:
'''
qa_data_qa_id: 결측치 0건
qa_data_instruction: 결측치 0건
qa_data_input_question: 결측치 0건
qa_data_input_answer: 결측치 0건
qa_data_input_follow_up_question: 결측치 0건
qa_data_output: 결측치 0건
consulting_consulting_category: 결측치 0건
'''

# 2. Embedding을 위한 GPU 연결 + RAG용 데이터 Embedding
#### (기존 커널에서 colab 커널로 변경)

- VSCode에 Google Colab 확장 설치 필요
- 커널 Colab > New Colab Server > GPU > T4 선택
- 8만건 임베딩을 진행해야 하기 때문에 GPU 활용 권장
- Colab 무료 티어의 경우 세션 시간 제한이 있어 분할 또는 중간 저장 로직 추가 필요

In [ ]:
# colab GPU(T4) 연결 확인
!nvidia-smi 

In [ ]:
# colab GPU(T4) 연결 확인 2
import torch

print("CUDA 사용 가능:", torch.cuda.is_available())
print("GPU 이름:", torch.cuda.get_device_name(0))
print("PyTorch 버전:", torch.__version__)

In [ ]:
# 작업 공간 확인용
# 현재의 작업 디렉토리와 맞지 않고 아래와 같이 출력된다면 Colab 원격 서버 임시 작업 공간인 것
# /content
# ['.config', 'sample_data']
# 이 경우, Colab 원격 서버로 데이터 옮겨주어야 함
import os

print(os.getcwd())
print(os.listdir("."))

In [ ]:
# Colab 원격 서버에 구글 드라이브 파일 다운
# * 구글 드라이브 파일 install 위해 "gdown" 라이브러리 추가 설치
# 민감한 데이터는 다운 후 드라이브에서 삭제하기
import gdown

file_id = "1IqNn72Fftb4KjHSZbPOrmeFWva7wLOs-"
output = "./dontalk/data/data.xlsx"

gdown.download(id=file_id, output=output, quiet=False)

In [ ]:
# 확인용
import os
print(os.listdir("./dontalk/data/"))

In [ ]:
# df 확인
import pandas as pd

file_path = "./dontalk/data/data.xlsx"
df = pd.read_excel(file_path)
df

In [ ]:
# 필요한 컬럼만 추출 + 스키마에 맞게 이름 변경
column_map = {
    "qa_data_qa_id": "qa_id",
    "qa_data_instruction": "instruction",
    "qa_data_input_question": "question",
    "qa_data_input_answer": "answer",
    "qa_data_input_follow_up_question": "follow_up_question",
    "qa_data_output": "output",
    "qa_data_full_source": "full_source",
    "consulting_consulting_category": "consulting_category",
    "consulting_consulting_topic": "consulting_topic",
    "qa_data_qa_topic": "qa_topic",
    "qa_data_consulting_purpose": "consulting_purpose",
}

df_selected = df[list(column_map.keys())].rename(columns=column_map).copy()
print(f"전체 행 수: {len(df_selected)}")
df_selected.head(3)

In [ ]:
# 결측치 확인용
required_cols = ["qa_id", "instruction", "question", "consulting_category"]
for col in required_cols:
    missing = df_selected[col].isna().sum()
    print(f"{col}: 결측치 {missing}건")

# 결측이 있다면 해당 행 제외
before = len(df_selected)
df_selected = df_selected.dropna(subset=required_cols)
after = len(df_selected)
print(f"\n제외된 행: {before - after}건, 최종 사용 행: {after}건")

### Embedding Test
- 코드 에러 확인 및 시간 체크용 100건 샘플 테스트

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 디바이스: {device}")

# 임베딩 모델 로드
model = SentenceTransformer("dragonkue/snowflake-arctic-embed-l-v2.0-ko", device=device)
print("모델 로드 완료")

In [ ]:
import time

df_sample = df_selected.head(100).copy()

def encode_texts(texts, label):
    if texts == df_sample["full_source"].tolist():
        b_size = 16
    else:
        b_size = 32

    start_time = time.time()
    embeddings = model.encode(
        texts, 
        batch_size=b_size, 
        show_progress_bar=True,
        normalize_embeddings=True,   # 코사인 유사도 검색을 위한 정규화
        convert_to_numpy=True,)
    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"[{label}] 임베딩 완료. shape={embeddings.shape}\n소요 시간: {elapsed_time:.2f}초")
    return embeddings

start = time.time()

emb_q_sample = encode_texts(df_sample["question"].tolist(), "embedding_q")
emb_i_sample = encode_texts(df_sample["instruction"].tolist(), "embedding_i")
emb_full_sample = encode_texts(df_sample["full_source"].tolist(), "embedding_full")

elapsed = time.time() - start


print(f"\n전체 임베딩 소요 시간: {elapsed:.2f}초")
print(f"8만 건 예상 소요 시간(단순 비례): 약 {elapsed * 800 / 60:.1f}분")

print(len(emb_q_sample))


In [ ]:
len(emb_q_sample)

### 80,000건 Embedding

In [ ]:
# from sentence_transformers import SentenceTransformer
# import torch

# device = "cuda" if torch.cuda.is_available() else "cpu"
# print(f"사용 디바이스: {device}")

# # 임베딩 모델 로드
# model = SentenceTransformer("dragonkue/snowflake-arctic-embed-l-v2.0-ko", device=device)
# print("모델 로드 완료")

In [ ]:
import os
import time
import pandas as pd
import numpy as np

CHUNK_SIZE = 8000          # 8만건 / 8000 = 10개 청크로 분할
CHECKPOINT_DIR = "./dontalk/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


def encode_texts(texts, batch_size):
    return model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=False,   # 청크마다 진행률 대신 청크 단위 로그로 대체
        normalize_embeddings=True,
        convert_to_numpy=True,
    )


n_chunks = (len(df_selected) + CHUNK_SIZE - 1) // CHUNK_SIZE
overall_start = time.time()

for chunk_idx in range(n_chunks):
    checkpoint_path = f"{CHECKPOINT_DIR}/chunk_{chunk_idx:03d}.parquet"

    # 이미 처리된 청크는 건너뛰기 (재실행/재개 가능하게)
    if os.path.exists(checkpoint_path):
        print(f"청크 {chunk_idx+1}/{n_chunks} 이미 처리됨, 건너뜀")
        continue

    start_idx = chunk_idx * CHUNK_SIZE
    end_idx = min(start_idx + CHUNK_SIZE, len(df_selected))
    df_chunk = df_selected.iloc[start_idx:end_idx].copy()

    t0 = time.time()
    emb_q = encode_texts(df_chunk["question"].tolist(), batch_size=32)
    emb_i = encode_texts(df_chunk["instruction"].tolist(), batch_size=32)
    emb_full = encode_texts(df_chunk["full_source"].tolist(), batch_size=16)
    elapsed = time.time() - t0

    df_chunk["embedding_q"] = list(emb_q)
    df_chunk["embedding_i"] = list(emb_i)
    df_chunk["embedding_full"] = list(emb_full)

    for col in ["embedding_q", "embedding_i", "embedding_full"]:
        df_chunk[col] = df_chunk[col].apply(lambda x: x.tolist())

    df_chunk.to_parquet(checkpoint_path, index=False)

    total_elapsed = time.time() - overall_start
    print(f"청크 {chunk_idx+1}/{n_chunks} 완료 ({end_idx - start_idx}건, {elapsed:.1f}초) "
          f"| 누적 시간: {total_elapsed/60:.1f}분")

print("\n전체 청크 처리 완료!")

In [ ]:
# 청크 하나로 병합
import glob

chunk_files = sorted(glob.glob(f"{CHECKPOINT_DIR}/chunk_*.parquet"))
df_final = pd.concat([pd.read_parquet(f) for f in chunk_files], ignore_index=True)
print(f"최종 병합 결과: {len(df_final)}건")

df_final.to_parquet("./dontalk/data/embeddings_output.parquet", index=False)
print("최종 파일 저장 완료: ./dontalk/data/embeddings_output.parquet")

In [ ]:
df_final.head(3)

In [ ]:
df_final.to_excel("./dontalk/data/embeddings_output.xlsx", index=False)

In [ ]:
# Colab 가상 경로 내 임베딩 데이터 로컬PC로 내려받기
# Colab에서 내 구글 드라이브 연결
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import shutil

shutil.copy('./dontalk/data/embeddings_output.parquet', '/content/drive/MyDrive/[CHAEN]dontalk/embeddings_output.parquet')

In [ ]:
shutil.copy('./dontalk/data/embeddings_output.xlsx', '/content/drive/MyDrive/[CHAEN]dontalk/embeddings_output.xlsx')

# 3. DB에 데이터 적재

### * load_to_postgres.py를 통해 터미널에서 진행

# 4. RAG TEST 
#### * 3가지 Method에 대해 검색 성능 확인

In [ ]:
# 임베딩 모델에서 기존 snowflake와 같이 한국어 모델 또한 prompt_name='query' 방식을 지원하는지 확인하기 위한 코드
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("dragonkue/snowflake-arctic-embed-l-v2.0-ko")
print(model.prompts)

In [ ]:
# --------------------------------------------------------
# Setting
# 라이브러리 로드 / DB 연결 / 모델 로드
# --------------------------------------------------------
import os
import psycopg2
from psycopg2.extras import RealDictCursor
from pgvector.psycopg2 import register_vector
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv()

POSTGRESQL_PORT = os.getenv("POSTGRESQL_PORT")
POSTGRESQL_USER = os.getenv("POSTGRESQL_USER")
POSTGRESQL_PASSWORD = os.getenv("POSTGRESQL_PASSWORD")
POSTGRESQL_DBNAME = os.getenv("POSTGRESQL_DBNAME")

DB_CONFIG = {
    "host": "localhost",
    "port": POSTGRESQL_PORT,
    "user": POSTGRESQL_USER,
    "password": POSTGRESQL_PASSWORD,
    "dbname": POSTGRESQL_DBNAME
}

conn = psycopg2.connect(**DB_CONFIG)
register_vector(conn)
cur = conn.cursor(cursor_factory=RealDictCursor)

model = SentenceTransformer("dragonkue/snowflake-arctic-embed-l-v2.0-ko")
print("DB 연결 및 모델 로드 완료")

In [ ]:
# --------------------------------------------------------
# RAG TEST - 코사인 유사도 기준 Top-3
# --------------------------------------------------------
def search_top3(user_query: str, embedding_column: str = "embedding_full"):
    # user_query: 사용자가 입력한 신규 질의 텍스트
    # embedding_column: embedding_q / embedding_i / embedding_full 중 선택

    
    # 1) 신규 질의 인코딩 (query 프롬프트 적용 - 저장된 데이터와 다르게 처리해야 함)
    #    > user_query의 경우 질의로 짧은 문장, embeddig_full의 경우 설명이 포함된 긴 문장
    #    > 때문에 최근 임베딩 모델들은 학습할 때부터 질문 앞에는 특별한 머리말(프롬프트)을 붙이도록 훈련됨
    #    > 따라서, user_query를 인코딩할 때는 query 프롬프트를 적용해야 함
    #             (모델이 머리말을 보고 답을 찾는 질문인지, 찾아질 자료인지를 구분하고, 질문 벡터가 그 질문에 답이 되는 문서 벡터 근처에 놓이도록 함)
    #     +) normalize_embeddings=True 옵션을 적용해야 코사인 유사도 검색이 가능함 (벡터 길이를 1로 맞추는 정규화 옵션)
    query_vector = model.encode(user_query, prompt_name="query", normalize_embeddings=True)


    # 2) 코사인 거리 기준 Top-3 검색
    #    (거리가 가까운 순서로 정렬)
    #     HNSW 인덱스는 vector_cosine_ops로 만든 HNSW가 있으면 자동으로 사용됨
    sql = f"""
        SELECT qa_id, full_source, consulting_category, consulting_topic, qa_topic, consulting_purpose,
               {embedding_column} <=> %s AS distance
        FROM financial_consulting_qa
        ORDER BY {embedding_column} <=> %s
        LIMIT 3;
    """

    # SQL 안에 %s가 두 번 등장하기 때문에 값도 두 번 전달해야 함
    # 첫 번째는 SELECT 절에서 거리 계산용, 두 번째는 ORDER BY 절에서 정렬용
    cur.execute(sql, (query_vector, query_vector))
    return cur.fetchall()   # 결과 3행을 리스트로 반환


def print_results(results):
    for i, row in enumerate(results, 1):
        print("---------------------------------------------------")
        print(f"[{i}순위] 거리: {row['distance']:.4f}")
        print(f"과거 질의응답:\n{row['full_source']}")
        print(f"metadata:\n- consulting_category={row['consulting_category']}\n- consulting_topic={row['consulting_topic']}\n- qa_topic={row['qa_topic']}\n- consulting_purpose={row['consulting_purpose']}")
        print("---------------------------------------------------")

In [ ]:
# user_query = input("질문을 입력하세요: ")

# for col in ["embedding_q", "embedding_i", "embedding_full"]:
#     print(f"사용자 질의: {user_query}")
#     print(f"\n{'='*60}")
#     print(f"[{col}] 기준 검색 결과")
#     print(f"{'='*60}")
#     results = search_top3(user_query, embedding_column=col)
#     print_results(results)

In [ ]:
# Method 별 따로 보기
# 1. embedding_q 기준 검색
user_query = input("질문을 입력하세요: ")

em_col = "embedding_q"
print(f"\n{'='*60}")
print(f"사용자 질의: {user_query}\n")
print(f"[{em_col}] 기준 검색 결과")
print(f"{'='*60}")
results = search_top3(user_query, embedding_column=em_col)
print_results(results)

In [ ]:
# 2. embedding_i 기준 검색
user_query = input("질문을 입력하세요: ")

em_col = "embedding_i"
print(f"\n{'='*60}")
print(f"사용자 질의: {user_query}\n")
print(f"[{em_col}] 기준 검색 결과")
print(f"{'='*60}")
results = search_top3(user_query, embedding_column=em_col)
print_results(results)

In [ ]:
# 3. embedding_full 기준 검색
user_query = input("질문을 입력하세요: ")

em_col = "embedding_full"
print(f"\n{'='*60}")
print(f"사용자 질의: {user_query}\n")
print(f"[{em_col}] 기준 검색 결과")
print(f"{'='*60}")
results = search_top3(user_query, embedding_column=em_col)
print_results(results)